In [1]:
import torch
from pathlib import Path
import torchaudio
from torch.utils.data import Dataset

In [2]:
print (torch.cuda.is_available())
print(torch.cuda.get_device_properties())
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.2f} GB')
print(f'CUDA Version: {torch.version.cuda}')

True
_CudaDeviceProperties(name='NVIDIA GeForce RTX 2060', major=7, minor=5, total_memory=6143MB, multi_processor_count=30, uuid=6e79eb6f-09fa-1fe4-df0e-cc6342c1db19, L2_cache_size=3MB)
VRAM: 6.00 GB
CUDA Version: 12.4


In [3]:
files = Path('../data/train_audio').rglob('*.ogg')
print(f'number of files in train_audio folder: {len(list(files))}')
import pandas as pd
train_df = pd.read_csv('../data/train.csv')
print(f'number of data rows in train.csv: {len(train_df)}')



number of files in train_audio folder: 35549
number of data rows in train.csv: 35549


In [4]:

train_df.head(10)

,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat
3,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/818781.m...,1161364/iNat818781.ogg,iNat
4,1161364,[],[],-22.7426,-46.8985,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/556514.m...,1161364/iNat556514.ogg,iNat
5,1161364,[],[],-22.0843,-47.7327,Guyalna cuta,Guyalna cuta,Insecta,1161364,Carlos Otávio Gussoni,cc-by-nc,0.0,https://static.inaturalist.org/sounds/868369.w...,1161364/iNat868369.ogg,iNat
6,1161364,[],[],-20.9848,-43.7609,Guyalna cuta,Guyalna cuta,Insecta,1161364,Pedro Cavalcante,cc-by-nc,0.0,https://static.inaturalist.org/sounds/842139.w...,1161364/iNat842139.ogg,iNat
7,1161364,[],[],-19.9674,-43.9895,Guyalna cuta,Guyalna cuta,Insecta,1161364,Pedro Cavalcante,cc-by-nc,0.0,https://static.inaturalist.org/sounds/840159.m...,1161364/iNat840159.ogg,iNat
8,1161364,[],[],-19.8713,-43.9607,Guyalna cuta,Guyalna cuta,Insecta,1161364,Alexandre S. Michelotto,cc0,0.0,https://static.inaturalist.org/sounds/1264238....,1161364/iNat1264238.ogg,iNat
9,1161364,[],[],-18.8358,-40.7354,Guyalna cuta,Guyalna cuta,Insecta,1161364,Vitor C. Dias Gonçalves,cc-by-nc,0.0,https://static.inaturalist.org/sounds/869958.m...,1161364/iNat869958.ogg,iNat


### Check that files in subfolders are the same as the entries in train.csv

In [ ]:
folders = [f for f in Path('../data/train_audio').iterdir() if f.is_dir()]
print(f'number of folders in train_audio: {len(folders)}')
for folder in folders:
    files_in_folder = [str(f.relative_to('../data/train_audio/')) for f in folder.rglob('*.ogg')]
    sub_df = train_df[train_df['primary_label']== folder.name]['filename']
    files_set = set(files_in_folder)
    sub_df_set = set(sub_df)
    missing_in_folder = sub_df_set - files_set
    missing_in_csv = files_set - sub_df_set
    if (len(missing_in_csv)!=0) or (len(missing_in_csv)!=0): 
        print(f'{folder}: In df but not in folder: ', missing_in_folder)
        print(f'{folder}: In folder but not in df: ', missing_in_csv)

number of folders in train_audio: 206


### Chunking the long files

In [ ]:
def chunk_long_file(file_path, output_dir, chunk_duration=5):
    # Load the audio file
    waveform, sample_rate = torchaudio.load(file_path)
    
    # Calculate the number of samples for the chunk duration
    chunk_samples = int(chunk_duration * sample_rate)
    
    # Get the total number of samples in the waveform
    total_samples = waveform.size(1)
    
    # Calculate the number of chunks
    num_chunks = total_samples // chunk_samples
    
    # Create output directory if it doesn't exist
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Split and save chunks
    for i in range(num_chunks):
        start_sample = i * chunk_samples
        end_sample = (i + 1) * chunk_samples
        chunk = waveform[:, start_sample:end_sample]
        
        # Save the chunk
        output_file = output_dir / f"{file_path.stem}_chunk_{i+1}.wav"
        torchaudio.save(output_file, chunk, sample_rate)
        print(f"Saved {output_file}")


In [20]:
reduced_df = train_df[['primary_label']].copy()
reduced_df['file_path'] = train_df.apply(lambda row: f"../data/train_audio/{row['filename']}", axis=1)
reduced_df['offset_sec'] = 0
reduced_df.head(5)

,primary_label,file_path,offset_sec
0,1161364,../data/train_audio/1161364/iNat1216197.ogg,0
1,1161364,../data/train_audio/1161364/iNat1114648.ogg,0
2,1161364,../data/train_audio/1161364/iNat810195.ogg,0
3,1161364,../data/train_audio/1161364/iNat818781.ogg,0
4,1161364,../data/train_audio/1161364/iNat556514.ogg,0


In [18]:
print(reduced_df.iloc[0, 1])

../data/train_audio/1161364/iNat1216197.ogg


In [ ]:

waveform, samplerate = torchaudio.load(reduced_df.iloc[0,1])
print(f'Samplerate: {samplerate} Hz')
print(f'Audio data shape: {waveform.shape}')
a = int((waveform.shape[1]-64)/samplerate) # -64 to remove the pad of silence sapmle added by OGG Vorbis when compressing audio blocks.
b = waveform.shape[1] % samplerate



Samplerate: 32000 Hz
Audio data shape: torch.Size([1, 576832])


In [65]:
import torchaudio
waveform, _ = torchaudio.load(filepath)
print(waveform.shape)

torch.Size([1, 576832])


In [ ]:
class BirdSoundDataset(Dataset):
    def __init__(self, df, audio_dir, transform=None):
        self.df = df
        self.audio_dir = Path(audio_dir)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file = self.audio_dir / row['filename']
        waveform, sample_rate = torchaudio.load(str(file))
        if self.transform:
            waveform = self.transform(waveform)
        label = row['label_encoded']
        return waveform, label
    
dataset = BirdDataset(train_df, '../data/train_audio')


6


55
